In [8]:
import json
import pickle
import string
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Σταθερό (hardcoded) absolute path του project.
# Αν τρέξεις σε άλλο μηχάνημα, άλλαξε ΜΟΝΟ αυτή τη γραμμή.
PROJECT_ROOT = Path("/Users/koukis/dev/personal/bigblue-academy/nlp_text_classification")
DATA_PATH = PROJECT_ROOT / "cleaned_data.csv"
MODELS_DIR = PROJECT_ROOT / "models"

TEXT_COL = "text_content"
LABEL_COL = "label"
RANDOM_STATE = 42

print("DATA_PATH:", DATA_PATH)
print("DATA_PATH υπάρχει;", DATA_PATH.exists())

DATA_PATH: /Users/koukis/dev/personal/bigblue-academy/nlp_text_classification/cleaned_data.csv
DATA_PATH υπάρχει; True


## 1. Φόρτωση δεδομένων

Διαβάζουμε το CSV. Το κείμενο είναι στη στήλη `text_content`, το target στη `label` (`human`/`ai`).

In [9]:
df = pd.read_csv(DATA_PATH)
print(f"{len(df)} γραμμές, στήλες: {list(df.columns)}")
df.head()

2000 γραμμές, στήλες: ['text_id', 'label', 'source_model', 'domain', 'text_content', 'topic_hint', 'word_count', 'avg_sentence_length', 'generation_method']


,text_id,label,source_model,domain,text_content,topic_hint,word_count,avg_sentence_length,generation_method
0,TXT_0001,human,human,social,can we talk about gene editing ethics for a se...,gene editing ethics,27,13.5,template+human_variation
1,TXT_0002,human,human,social,update on election integrity concerns: it's co...,election integrity concerns,20,10.0,template+human_variation
2,TXT_0003,ai,gemini-2.0,news,Analysts are closely watching developments rel...,climate change adaptation strategies,39,13.0,style_simulation
3,TXT_0004,human,human,academic,This paper examines genomic research breakthro...,genomic research breakthroughs,49,16.3,template+human_variation
4,TXT_0005,ai,gpt-4o,academic,Existing literature on student debt crisis has...,student debt crisis,42,10.8,style_simulation


## 2. Labels:  human → 0,  ai → 1

Ρητή αντιστοίχιση των string labels σε ακέραιους (`human`=0, `ai`=1).
Κρατάμε το `label_mapping` ώστε ο modeler να ξέρει την αντιστοίχιση.

**Σημ. — imbalance:** ~66% human / 34% ai (να το έχει υπόψη ο modeler στο μοντέλο).

In [10]:
# Ρητή αντιστοίχιση (ΟΧΙ LabelEncoder, που θα έδινε αλφαβητικά ai=0/human=1):
#   human -> 0,  ai -> 1
label_mapping = {"human": 0, "ai": 1}
y = df[LABEL_COL].map(label_mapping).to_numpy()

print("label_mapping:", label_mapping)
print("κατανομή κλάσεων:", {k: int((y == v).sum()) for k, v in label_mapping.items()})

label_mapping: {'human': 0, 'ai': 1}
κατανομή κλάσεων: {'human': 1334, 'ai': 666}


## 3. Text → embeddings (HuggingFace `BAAI/bge-m3`)

Διανυσματοποιούμε το `text_content` με το pre-trained model `bge-m3` (1024 διαστάσεις).
- **Δεν** κάνουμε χειροκίνητο preprocessing — το transformer έχει δικό του tokenizer.
- `normalize_embeddings=True` → unit vectors (καλό για cosine & για το concat με τα στυλομετρικά).
- Πρώτη φορά κατεβάζει ~2.2GB (μετά cached στο `~/.cache/huggingface`). Σε CPU με 2.000 σύντομα κείμενα τρέχει άνετα.

_Lighter fallback αν αργεί:_ `"sentence-transformers/all-MiniLM-L6-v2"` (384-dim).

In [11]:
import torch
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "BAAI/bge-m3"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device = {device}")

model = SentenceTransformer(EMBED_MODEL, device=device)

emb = model.encode(
    df[TEXT_COL].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)
emb = np.asarray(emb)
print("embeddings shape:", emb.shape)

device = cpu


KeyboardInterrupt: 

## 4. Στυλομετρικά features + σύνθεση feature matrix X

Φτιάχνουμε 4 στυλομετρικά (το exploration notebook έδειξε ότι διαχωρίζουν human/ai):
`word_count`, `avg_sentence_length` (από το CSV) + `word_diversity` (TTR) και `punctuation_density`
(τα υπολογίζουμε από το `text_content`).

Στη συνέχεια ενώνουμε **embeddings + στυλομετρικά** σε ένα πίνακα `X`. Κάνουμε **scale μόνο τα
στυλομετρικά** με `StandardScaler` (τα bge-m3 dims είναι ήδη unit-normalized· το `word_count` ~20-50
θα κυριαρχούσε αλλιώς). Σώζουμε τον scaler για μελλοντικά δεδομένα.

In [ ]:
# --- Στυλομετρικά features ---
punct = set(string.punctuation)

def ttr(text):
    # Type-Token Ratio: λεξιλογική ποικιλία = μοναδικά tokens / σύνολο tokens.
    tokens = str(text).lower().split()
    return len(set(tokens)) / len(tokens) if tokens else 0.0

def punct_density(text, wc):
    # πλήθος σημείων στίξης / (word_count + 1)  (+1 για αποφυγή διαίρεσης με 0)
    n_punct = sum(ch in punct for ch in str(text))
    return n_punct / (wc + 1)

stylo = np.column_stack([
    df["word_count"].to_numpy(dtype=float),
    df["avg_sentence_length"].to_numpy(dtype=float),
    df[TEXT_COL].apply(ttr).to_numpy(dtype=float),
    np.array([punct_density(t, wc) for t, wc in zip(df[TEXT_COL], df["word_count"])], dtype=float),
])

# --- Σύνθεση X: embeddings + scaled στυλομετρικά ---
scaler = StandardScaler()
stylo_scaled = scaler.fit_transform(stylo)

X = np.hstack([emb, stylo_scaled])
print(f"X shape: {X.shape}  ({emb.shape[1]} embeddings + {stylo.shape[1]} stylometrics)")
print(f"y shape: {y.shape}")

## 6. Validation — UMAP 2D

Οπτικός έλεγχος: αν τα embeddings φέρουν σήμα human/ai, τα δύο χρώματα θα τείνουν να χωρίζονται.
(Μόνο για επισκόπηση — δεν μπαίνει στο X.)

In [ ]:
import umap

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=RANDOM_STATE)
emb_2d = reducer.fit_transform(emb)

plt.figure(figsize=(8, 6))
for cls, idx in label_mapping.items():
    mask = y == idx
    plt.scatter(emb_2d[mask, 0], emb_2d[mask, 1], s=8, alpha=0.5, label=cls)
plt.legend(); plt.title("UMAP των bge-m3 embeddings (χρώμα = label)")
plt.xlabel("UMAP-1"); plt.ylabel("UMAP-2"); plt.show()

## 6. Τελικό αποτέλεσμα: `X` (vectors) + `y` (labels)

- `X` → το feature matrix (embeddings + στυλομετρικά)
- `y` → τα labels (human=0, ai=1)

Φτιάχνουμε και ένα ενιαίο `df_features` (vectors + στήλη `label`) για ευκολία, και σώζουμε τα artifacts
στο `models/` ώστε ο modeler να τα φορτώνει χωρίς να ξανατρέχει το embedding βήμα.

In [ ]:
# Ενιαίο DataFrame: μία στήλη ανά διάσταση του vector (f0..f1027) + στήλη label.
df_features = pd.DataFrame(X, columns=[f"f{i}" for i in range(X.shape[1])])
df_features["label"] = y
print("df_features shape:", df_features.shape)
display(df_features.head())

# --- Αποθήκευση artifacts στο models/ (ΧΩΡΙΣ split) ---
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# X & y σε .npz (συμπιεσμένο, γρήγορο φόρτωμα: d = np.load(...); d["X"], d["y"])
np.savez_compressed(MODELS_DIR / "features.npz", X=X, y=y)
# Το ίδιο και ως CSV, αν το προτιμά ο modeler
df_features.to_csv(MODELS_DIR / "features.csv", index=False)
# Mapping κλάσεων + ο scaler των στυλομετρικών (για μελλοντικά δεδομένα)
with open(MODELS_DIR / "label_mapping.json", "w") as f:
    json.dump(label_mapping, f, indent=2)
with open(MODELS_DIR / "scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Έτοιμα τα artifacts στο", MODELS_DIR)
print(" - features.npz  (X, y)")
print(" - features.csv  (df_features: vectors + label)")
print(" - label_mapping.json, scaler.pkl")